In [1]:
from hydra import compose, initialize
from omegaconf import OmegaConf
import json
from pathlib import Path
with initialize(config_path="dev/whole_body_benchmark/configs"):
    cfg = compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

from pathlib import Path
import glob
import os
img_base_path = Path(cfg.paths.nako_dir) / "links"
mask_base_path = Path(cfg.paths.nako_dir) / "data"
img_target = "30/3D_GRE_TRA_4/3D_GRE_TRA_W_COMPOSED*.nii"
mask_target = "30/opportunistic-screening/seg.nii.gz"

/tmp/ipykernel_65/3643711350.py:5: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize(config_path="dev/whole_body_benchmark/configs"):


paths:
  nako_dir: /nfs/data/nii/data0/GNC/GNC_759
  nnUNet_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/nnunet/
  results_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/results/
  data_dir: /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/
  oppscreen_dir: /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/



In [2]:
# list all subjects that have an image and mask

subjects = [e.name for e in os.scandir(img_base_path) if e.is_dir()]
subjects_with_mask = []
for subject in subjects:
    mask_path = os.path.join(mask_base_path, subject, mask_target)
    if os.path.exists(mask_path):
        subjects_with_mask.append(subject)
print(f"subjects with image : {len(subjects)}")
print(f"subjects with image and mask : {len(subjects_with_mask)}")

subjects with image : 31006
subjects with image and mask : 966


In [4]:
import nibabel as nib
import numpy as np
from multiprocessing import Pool

def get_labels(subject):
    mask_path = os.path.join(mask_base_path, subject, mask_target)
    mask = nib.load(mask_path)
    mask_data = mask.get_fdata()
    return subject, np.unique(mask_data)

with Pool(18) as pool:
    results = pool.map(get_labels, subjects_with_mask)

labels_by_subject = dict(results)

In [10]:
labels_by_subject

{'100121': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100156': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100511': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100485': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100566': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100588': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,
        14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.]),
 '100846': array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 13.,

In [5]:
import random

random.seed(42)
shuffled = subjects_with_mask.copy()
random.shuffle(shuffled)
split = int(0.7 * len(shuffled))
splits = {'train': shuffled[:split], 'test': shuffled[split:]}
print(f"train: {len(splits['train'])}, test: {len(splits['test'])}")

train: 676, test: 290


In [11]:
train_subset_sizes = [10, 20, 30, 50, 100, 200, 300]

for n in train_subset_sizes:
    splits[f'train_{n}'] = splits['train'][:n]

print({k: len(v) for k, v in splits.items()})

{'train': 676, 'test': 290, 'train_10': 10, 'train_20': 20, 'train_30': 30, 'train_50': 50, 'train_100': 100, 'train_200': 200, 'train_300': 300}


In [12]:
out_path = Path(cfg.paths.data_dir) / 'splits_966.json'

with open(out_path, 'w') as f:
    json.dump(splits, f)
print(f'Saved to {out_path}')
"""
with open(out_path, 'r') as f:
    splits_saved = json.load(f)
"""

Saved to /nfs/data/nii/data0/GNC/Analysis/GNC_759/ANALYSIS_whole_body_benchmark/data/splits_966.json


"\nwith open(out_path, 'r') as f:\n    splits_saved = json.load(f)\n"

In [ ]:
# copy images and masks to whole_body_benchmark project


import shutil
from multiprocessing import Pool

os.makedirs(os.path.join(cfg.paths.oppscreen_dir, "images"), exist_ok=True)
os.makedirs(os.path.join(cfg.paths.oppscreen_dir, "masks"), exist_ok=True)

def copy_subject(subject):
    img_pattern = os.path.join(str(img_base_path), subject, img_target)
    img_matches = glob.glob(img_pattern)
    if not img_matches:
        return subject, "skipped (no image)"
    mask_src = os.path.join(mask_base_path, subject, mask_target)
    for img_src in img_matches:
        #orig_name = Path(img_src).name
        img_dst = os.path.join(cfg.paths.oppscreen_dir, "images", f"{subject}_img.nii")
        shutil.copy(img_src, img_dst)
    mask_dst = os.path.join(cfg.paths.oppscreen_dir, "masks", subject + "_mask.nii.gz")
    shutil.copy(mask_src, mask_dst)
    return subject, f"ok ({len(img_matches)} images)"



[1/10] 100058: ok (1 images)
[2/10] 100381: ok (1 images)
[3/10] 100018: ok (1 images)
[4/10] 100360: ok (1 images)
[5/10] 100137: ok (1 images)
[6/10] 100736: ok (1 images)
[7/10] 100452: ok (1 images)
[8/10] 100901: ok (1 images)
[9/10] 100537: ok (1 images)
[10/10] 100573: ok (1 images)


In [22]:
# copy train and test subjects to oppscreen dir
for split_name in ['train', 'test']:
    subjects = splits[split_name]
    print(f"Copying {len(subjects)} {split_name} subjects...")
    with Pool(12) as pool:
        for i, (subject, status) in enumerate(pool.imap_unordered(copy_subject, subjects), 1):
            print(f"[{i}/{len(subjects)}] {subject}: {status}")


Copying 676 train subjects...


[1/676] 100381: ok (1 images)
[2/676] 100573: ok (1 images)
[3/676] 100401: ok (1 images)
[4/676] 100058: ok (1 images)
[5/676] 100384: ok (1 images)
[6/676] 100018: ok (1 images)
[7/676] 100452: ok (1 images)
[8/676] 100137: ok (1 images)
[9/676] 100901: ok (1 images)
[10/676] 100736: ok (1 images)
[11/676] 100360: ok (1 images)
[12/676] 100537: ok (1 images)
[13/676] 100626: ok (1 images)
[14/676] 100271: ok (1 images)
[15/676] 100311: ok (1 images)
[16/676] 100550: ok (1 images)
[17/676] 100680: ok (1 images)
[18/676] 100231: ok (1 images)
[19/676] 100540: ok (1 images)
[20/676] 100997: ok (1 images)
[21/676] 100842: ok (1 images)
[22/676] 100840: ok (1 images)
[23/676] 100959: ok (1 images)
[24/676] 100131: ok (1 images)
[25/676] 100970: ok (1 images)
[26/676] 100310: ok (1 images)
[27/676] 100119: ok (1 images)
[28/676] 100746: ok (1 images)
[29/676] 100195: ok (1 images)
[30/676] 100727: ok (1 images)
[31/676] 100927: ok (1 images)
[32/676] 100926: ok (1 images)
[33/676] 100965: 

In [ ]:
# import images and masks into Nora project
project_name = "camaret___whole_body_benchmark"
img_dir  = os.path.join(cfg.paths.oppscreen_dir, "images")
mask_dir = os.path.join(cfg.paths.oppscreen_dir, "masks")

img_regex  = r"(?<patients_id>\d+)_img\.nii"
mask_regex = r"(?<patients_id>\d+)_mask\.nii\.gz"

print("Importing images...")
#!nora -p {project_name} --importfiles {img_dir} '{img_regex}'
print(f"nora -p {project_name} --importfiles {img_dir} '{img_regex}'")

print("Importing masks...")
#!nora -p {project_name} --importfiles {mask_dir} '{mask_regex}'
print(f"nora -p {project_name} --importfiles {mask_dir} '{mask_regex}'")

#print("Tagging masks...")
#!nora -p {project_name} --addtag mask -s "* *_mask.nii.gz"
#print(f"nora -p {project_name} --addtag mask -s '* *_mask.nii.gz'")

Importing images...
nora -p camaret___whole_body_benchmark --importfiles /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/images '(?<patients_id>\d+)_img\.nii'
Importing masks...
nora -p camaret___whole_body_benchmark --importfiles /nfs/data/nii/data1/Analysis/camaret___whole_body_benchmark/ANALYSIS_ana001/data/oppscreen/masks '(?<patients_id>\d+)_mask\.nii\.gz'
Tagging masks...
nora -p camaret___whole_body_benchmark --addtag mask -s '* *_mask.nii.gz'


In [14]:
# tag patients with train/test/train_N in Nora
import subprocess
project_name = "camaret___whole_body_benchmark"

for split_name, subjects in splits.items():
    print(f"Tagging {len(subjects)} patients as '{split_name}'...")
    for subject in subjects:
        result = subprocess.run(
            ["nora", "-p", project_name, "--addtag_patient", split_name, subject],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  ERROR {subject}: {result.stderr.strip()}")
    print(f"  Done.")

Tagging 676 patients as 'train'...
  Done.
Tagging 290 patients as 'test'...
  Done.
Tagging 10 patients as 'train_10'...
  Done.
Tagging 20 patients as 'train_20'...
  Done.
Tagging 30 patients as 'train_30'...
  Done.
Tagging 50 patients as 'train_50'...
  Done.
Tagging 100 patients as 'train_100'...
  Done.
Tagging 200 patients as 'train_200'...
  Done.
Tagging 300 patients as 'train_300'...
  Done.
